# 34. Is the task removed, or only blocked?This notebook runs one measurement and answers one question. It trains nothing.### The questionUnlearning trains the shared generator so that the forgotten task's own code producesnear-zero weights. That code then stops working, which we have measured many times.But the generator is shared. The paper's own appendix shows that before unlearning, an**unused random task code** still reaches a task through the generator, scoring up to80.91 percent against 10 percent chance. In other words, a task is reachable from morethan one address.Nobody has run that measurement after unlearning. That is what this notebook does.### Why the answer matters| What an unused code scores after unlearning | What it means || --- | --- || Well above chance | Unlearning blocked one route. The skill is still in the generator and still reachable. The paper's removal claim is not supported. || At chance | The skill really was taken out of the generator. The method does what it claims. |### The control that keeps this honestUnlearning moves the shared generator, so it might damage reading through random codesfor **every** task, not just the forgotten one. If that happened, a drop to chance on theforgotten task would mean nothing.So we run the same probe on a task that was kept. If random codes still read the kepttask well after unlearning, and only the forgotten task falls to chance, the removal isspecific and real. If random codes stop working everywhere, the probe cannot decideanything and we say so.### What this notebook does not doNo training, no recovery, no fine-tuning. Every number below comes from forward passesthrough weights the generator produces on the spot. That is the point: our earlierrecovery probe trained the generator, and training can rebuild a skill, so it could neverseparate "still there" from "learned again".

## 1. Load the fixed code

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading this fixed code revision.")
if IN_COLAB:
    REPO = Path("/content/cark-paired-study")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    if subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--untracked-files=no"], text=True).strip():
        raise RuntimeError("Checkout has edits. Use a fresh runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", CODE_VERSION], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a CARK checkout.")
    actual = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
    implementation = ["uncle", "pyproject.toml", "requirements.txt"]
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain", "--", *implementation], text=True).strip()
    # A later documentation-only commit may contain the same fixed implementation.
    differences = subprocess.check_output(["git", "-C", str(REPO), "diff", CODE_VERSION, actual, "--", *implementation], text=True).strip()
    if dirty or differences:
        raise RuntimeError("Use a clean checkout of CODE_VERSION, or open in Colab. Local training edits are preserved.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies before running this notebook.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))
from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng, run_request
# The progress panel only draws bars, so it cannot change a result. It
# therefore sits outside the pinned implementation, in scripts/, and this
# checkout is detached to CODE_VERSION, which predates it. Try the checkout
# first, then main, then fall back to ordinary bars rather than failing.
sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError:
    try:
        tools = Path.cwd() / '_tools'
        tools.mkdir(exist_ok=True)
        subprocess.run(['git', '-C', str(REPO), 'fetch', '--quiet', 'origin', 'main'], check=True)
        (tools / 'dashboard.py').write_bytes(subprocess.check_output(
            ['git', '-C', str(REPO), 'show', 'origin/main:scripts/dashboard.py']))
        sys.path.insert(0, str(tools))
        from dashboard import Dashboard
        print('Progress panel fetched from main.')
    except Exception as problem:
        print('Progress panel unavailable, using plain bars:', problem)

        class Dashboard:
            """Just enough of the panel to keep the notebook running."""

            def __init__(self, *args, **kwargs): pass
            def __enter__(self): return self
            def __exit__(self, *args): return False
            def note(self, text): print(text, flush=True)

            def bar(self, total, description, unit='step'):
                from uncle.telemetry import progress_bar
                return progress_bar(total, description)

import torch
print("Fixed implementation revision:", CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"
SEED = 0

# The task to forget. Task 14 is last in order 01, so nothing was learned after
# it and it is at or near its best accuracy when unlearning starts. Our other
# probes used it for the same reason, so the numbers here line up with those.
FORGET_TASK = "14"

# Two stopping points, the same two the rest of the study uses. 6 is where our
# accuracy trace first puts the task at chance. 100 is the paper's own setting.
# Running both answers a second question for free: if an unused code still reads
# the task after 6 steps but not after 100, then the extra steps are doing the
# real removal and the paper's stopping rule matters more than it looks.
#
# Each stopping point rebuilds the model from the saved checkpoint and unlearns
# from scratch, so the 100-step run is never a continuation of the 6-step one.
UNLEARN_BUDGETS = [6, 100]

# How many unused random codes to try. One code could be unlucky. Five gives a
# spread, and they are cheap because nothing is trained.
PROBE_CODES = 5

# Tasks the model never saw. Needed only so the dataset builder prepares them;
# this notebook does not probe them. Kept here so the loading cell below, which
# is shared with notebook 24, works unchanged.
UNSEEN_TASKS = ["2", "4", "6"]

SAVE_TO_R2 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID`, and `R2_SECRET_ACCESS_KEY` through Colab Secretsor environment variables.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the study and the source modelThe source model has learned all eight tasks of the order and has forgotten nothing yet.Every measurement below starts from it.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
path = store.download(PREFIX + "/study.json", CACHE / "cache")
plan = json.loads(path.read_text())
if plan["code_version"] != CODE_VERSION:
    raise ValueError("This study requires a different fixed code revision.")
from uncle.learning_diagnostic import file_hash
if file_hash(REPO / "uncle/task_partition.json") != plan["partition_sha256"]:
    raise ValueError("Class partition differs from the saved study.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
NEW_TASK = plan["new_tasks"][0]
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

def fetch_tasks(include):
    """Build the task datasets, clearing a broken download if there is one."""
    import shutil, zipfile
    archive = Path(DATA_ROOT).parent / "tiny-imagenet-200.zip"
    for attempt in (1, 2):
        try:
            return build_tasks(root=DATA_ROOT,
                               partition_path=REPO / "uncle/task_partition.json",
                               partition_seed=42, include=include, download=True)
        except (zipfile.BadZipFile, EOFError, OSError) as problem:
            if attempt == 2:
                raise
            print("The Tiny ImageNet download is unusable:", problem)
            if archive.exists():
                head = archive.open("rb").read(4)
                print(f"   {archive} is {archive.stat().st_size} bytes, starts with {head!r}")
                archive.unlink()
            shutil.rmtree(Path(DATA_ROOT), ignore_errors=True)
            print("   Cleared it. Downloading again.")

tasks = fetch_tasks(ORDER_TASKS + tuple(UNSEEN_TASKS))

source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
branch_job = next(j for j in ALL_JOBS
                  if j["id"] == f"branches/{ORDER}/seed_{SEED}/forget_{FORGET_TASK}/learn_{NEW_TASK}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

REQUESTS = tuple(("learn", t) for t in ORDER_TASKS)
REQUESTS += tuple(tuple(op) for op in branch_job["operations"])
PROTECT = [t for t in state["seen"] if t != FORGET_TASK]
print("Source:", source_job["id"], "| forgetting task", FORGET_TASK)

## 5. Pick the kept task used as a controlThe control task must be one the model learned and keeps. We take the last kept task inthe order, because it is the most recently learned and so the least damaged by ordinaryforgetting. That gives the control the best chance of reading well, which is what we want:if even this task cannot be read through a random code after unlearning, the probe istelling us about the generator in general and not about removal.

In [ ]:
RETAIN_TASK = [t for t in ORDER_TASKS if t != FORGET_TASK][-1]
chance = 100.0 / CLASSES
print("Forgotten task:", FORGET_TASK)
print("Kept control task:", RETAIN_TASK)
print("Chance level:", chance, "percent")

## 6. Measuring`inspect` is the same measurement the rest of the study uses. It scores the weights thatone task code produces, on another task's test images, and also reports:- **input sensitivity**: how much the ten output scores move between different images. A  model that has stopped reading its input reads near zero here.- **classes used**: how many of the ten classes it actually predicts at least once. One  class means it has collapsed onto a single answer, which scores exactly chance on a  balanced set without having forgotten anything.- **top class share**: the fraction of all predictions that went to the single most common  class. 0.1 is an even spread, 1.0 is total collapse.Those three matter here as much as accuracy. A random code that reads a task at 40 percentwhile using all ten classes is genuinely reading it. One that happens to score 12 percentwhile answering a single class is not.

In [ ]:
from torch.func import functional_call
from torch.utils.data import DataLoader, Subset
import torch.nn.functional as F

def inspect(hypernet, target, trainer, code_task, data_task):
    """Score the weights produced by `code_task` on `data_task`'s images.

    The two differ in the fresh-start and stats-only arms, where a brand new
    code is trained on the forgotten task's data, so its weights must still be
    judged on that task's images.
    """
    # Evaluation mode matters more than usual here. In training mode
    # BatchNorm normalises using each batch's own statistics, which rescales a
    # nearly dead signal back up to unit size and hides exactly the collapse
    # this notebook is measuring. The previous mode is put back afterwards so
    # a measurement taken mid-training does not stop the training.
    was_training = hypernet.training
    hypernet.eval(); target.eval()
    weights = hypernet.weights_for(code_task)
    buffers = {n: v.clone() for n, v in trainer.task_buffers[code_task].items()}
    loader = DataLoader(tasks[data_task]["test"], batch_size=EVAL_BATCH)
    correct = total = 0
    lowest = highest = None
    # How often each class is predicted, over the whole test set. This is what
    # separates a collapsed model from a guessing one: both score 10%, but a
    # collapsed model puts every prediction in one or two bins.
    predicted = torch.zeros(CLASSES, dtype=torch.long)
    with torch.no_grad():
        for images, labels in loader:
            scores = functional_call(target, (weights, buffers),
                                     (images.to(trainer.device),), strict=True)
            guesses = scores.argmax(1)
            correct += (guesses == labels.to(trainer.device)).sum().item()
            total += labels.numel()
            predicted += torch.bincount(guesses.cpu(), minlength=CLASSES)
            low, high = scores.min(dim=0).values, scores.max(dim=0).values
            lowest = low if lowest is None else torch.minimum(lowest, low)
            highest = high if highest is None else torch.maximum(highest, high)
    if was_training:
        hypernet.train(); target.train()
    share = predicted.double() / max(int(predicted.sum()), 1)
    return {"accuracy": 100.0 * correct / total,
            "input_sensitivity": (highest - lowest).max().item(),
            "classes_used": int((predicted > 0).sum()),
            "top_class_share": float(share.max())}

def sample_indices(dataset, per_class, seed):
    """An equal number of images from each class, chosen reproducibly.

    ClassTask keeps its labels in `.targets`, so this costs nothing: no image
    is opened just to find out which class it belongs to.
    """
    generator = torch.Generator().manual_seed(seed)
    chosen = []
    for label in range(CLASSES):
        here = [i for i, value in enumerate(dataset.targets) if value == label]
        order = torch.randperm(len(here), generator=generator)[:per_class]
        chosen.extend(here[int(i)] for i in order)
    return chosen

## 7. The probeFor each stopping point the model is rebuilt from the saved checkpoint, measured, unlearned,and measured again. The same five random codes are used before and after, so any change isthe unlearning and not the codes.Each random code is given the target task's stored BatchNorm statistics. This matches thepaper's own baseline, which replaces the task embedding and changes nothing else. Unlearningdoes not touch those statistics, so they are the same before and after.

In [ ]:
import copy, gc

def build(burn_in_steps):
    # burn_in is what run_request actually uses as the number of unlearning
    # steps: `steps = config.burn_in`. forget_interval only controls how
    # often the boundary callback fires, so passing the stopping point there
    # did not change the step count. Until 28 October this read
    # max(UNLEARN_BUDGETS), so every arm was unlearned for 100 steps and the
    # two stopping points were the same model. It now takes the stopping
    # point as an argument.
    """A fresh model restored from the saved source checkpoint."""
    config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                       "requests": REQUESTS, "burn_in": burn_in_steps})
    target = build_target(config)
    hypernet = HyperNetwork(target, config)
    trainer = UnCLe(hypernet, config, target, tasks, progress=True)
    checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
    restore_rng(state["all_rng"])
    return config, target, hypernet, trainer

def through_random_codes(hypernet, target, trainer, data_task):
    """Read one task through unused random codes, with no training at all.

    This is the paper's appendix baseline. A brand new code is added, given the
    target task's own BatchNorm statistics, scored, and then removed again so it
    cannot affect the next measurement.
    """
    buffers = trainer.task_buffers[data_task]
    readings = []
    for index in range(PROBE_CODES):
        name = f"probe_{data_task}_{index}"
        if name in hypernet.task_codes:
            del hypernet.task_codes[name]
        # add_task draws the code from a stream keyed by the code's own name
        # and the config seed, not from global randomness. So the same name
        # always gives the same 32 numbers: code 0 here is the same vector
        # before and after unlearning, and at both stopping points.
        hypernet.add_task(name)
        trainer.task_buffers[name] = {n: v.clone() for n, v in buffers.items()}
        readings.append(inspect(hypernet, target, trainer, name, data_task))
        del hypernet.task_codes[name]
        del trainer.task_buffers[name]
    return readings

def mean_accuracy(readings):
    return sum(r["accuracy"] for r in readings) / len(readings)

def show(label, readings):
    scores = ", ".join(f"{r['accuracy']:.1f}" for r in readings)
    classes = ", ".join(str(r["classes_used"]) for r in readings)
    print(f"    {label:<34} mean {mean_accuracy(readings):5.1f}%   "
          f"each [{scores}]   classes used [{classes}]")

results = {}

for unlearn_steps in UNLEARN_BUDGETS:
    print(f"\n{'=' * 74}\nSTOPPING POINT: {unlearn_steps} UNLEARNING STEPS\n{'=' * 74}")

    # Free the previous stopping point's model first. The generator is large and
    # holding two of them is what runs a smaller GPU out of memory.
    for name in ("hypernet", "trainer", "target", "config"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    config, target, hypernet, trainer = build(unlearn_steps)

    print("  BEFORE unlearning")
    before = {
        "own_code_forget": inspect(hypernet, target, trainer, FORGET_TASK, FORGET_TASK),
        "random_codes_forget": through_random_codes(hypernet, target, trainer, FORGET_TASK),
        "random_codes_retain": through_random_codes(hypernet, target, trainer, RETAIN_TASK),
    }
    print(f"    {'task ' + FORGET_TASK + ' own code':<34} "
          f"     {before['own_code_forget']['accuracy']:5.1f}%")
    show(f"task {FORGET_TASK} through random codes", before["random_codes_forget"])
    show(f"task {RETAIN_TASK} through random codes", before["random_codes_retain"])

    # The forgetting operation itself, unchanged, straight from the study code.
    run_request(trainer, "forget", FORGET_TASK, PROTECT, None,
                lambda row: None, lambda active: None,
                forget_interval=unlearn_steps, evaluation_interval=10 ** 9)

    print("  AFTER unlearning")
    after = {
        "own_code_forget": inspect(hypernet, target, trainer, FORGET_TASK, FORGET_TASK),
        "random_codes_forget": through_random_codes(hypernet, target, trainer, FORGET_TASK),
        "random_codes_retain": through_random_codes(hypernet, target, trainer, RETAIN_TASK),
    }
    print(f"    {'task ' + FORGET_TASK + ' own code':<34} "
          f"     {after['own_code_forget']['accuracy']:5.1f}%")
    show(f"task {FORGET_TASK} through random codes", after["random_codes_forget"])
    show(f"task {RETAIN_TASK} through random codes", after["random_codes_retain"])

    results[unlearn_steps] = {"before": before, "after": after}

## 8. DecideThe rule is written down before the numbers are read, so the answer is not chosen afterthe fact.- A reading counts as **above chance** when it beats the chance level by more than 5 points.  Our repeat runs move accuracy by up to a few points, so 5 is comfortably outside that.- The forgotten task's own code is expected at chance after unlearning. If it is not, the  unlearning did not finish and nothing else on this page can be read.Three outcomes, and each one says something different.

In [ ]:
MARGIN = 5.0   # points above chance before a reading counts as real

print(f"chance = {chance:.1f}%   a reading counts as above chance above "
      f"{chance + MARGIN:.1f}%\n")

header = f"{'steps':>12} | {'own code':>9} | {'forget via random':>18} | {'retain via random':>18}"
print(header)
print("-" * len(header))
for unlearn_steps in UNLEARN_BUDGETS:
    run = results[unlearn_steps]
    for stage in ("before", "after"):
        row = run[stage]
        print(f"{str(unlearn_steps) + ' ' + stage:>12} | "
              f"{row['own_code_forget']['accuracy']:8.1f}% | "
              f"{mean_accuracy(row['random_codes_forget']):17.1f}% | "
              f"{mean_accuracy(row['random_codes_retain']):17.1f}%")

print()
for unlearn_steps in UNLEARN_BUDGETS:
    run = results[unlearn_steps]
    own_after = run["after"]["own_code_forget"]["accuracy"]
    forget_after = mean_accuracy(run["after"]["random_codes_forget"])
    retain_after = mean_accuracy(run["after"]["random_codes_retain"])
    forget_before = mean_accuracy(run["before"]["random_codes_forget"])
    retain_before = mean_accuracy(run["before"]["random_codes_retain"])

    print(f"--- {unlearn_steps} unlearning steps ---")

    if own_after > chance + MARGIN:
        print("  INVALID. The forgotten task's own code still works, so unlearning did")
        print(f"  not finish. Own-code accuracy is {own_after:.1f}% against chance {chance:.1f}%.")
        continue

    if forget_after > chance + MARGIN:
        print("  BLOCKED, NOT REMOVED.")
        print(f"  An unused random code still reads task {FORGET_TASK} at "
              f"{forget_after:.1f}%, against chance {chance:.1f}%,")
        print(f"  while the task's own code sits at {own_after:.1f}%. Unlearning closed one")
        print("  route and left the skill reachable from another. This is the result that")
        print("  would support a paper.")
    elif retain_after > chance + MARGIN:
        print("  REMOVED, AND SPECIFICALLY SO.")
        print(f"  Random codes read task {FORGET_TASK} at only {forget_after:.1f}% after")
        print(f"  unlearning, down from {forget_before:.1f}%, while the kept task "
              f"{RETAIN_TASK} still")
        print(f"  reads at {retain_after:.1f}%. The generator lost this task and kept the")
        print("  others. The method does what it claims, and the removal angle closes.")
    else:
        print("  CANNOT DECIDE.")
        print(f"  After unlearning, random codes read nothing: task {FORGET_TASK} at "
              f"{forget_after:.1f}% and")
        print(f"  kept task {RETAIN_TASK} at {retain_after:.1f}%, down from "
              f"{retain_before:.1f}%. Reading through an")
        print("  unused code stopped working everywhere, so this probe cannot separate")
        print("  removal from general damage to the generator. Report it as inconclusive.")
    print()

## 9. SaveThe summary goes to R2 beside the other diagnostics, so the numbers can be quoted laterwithout rerunning anything.

In [ ]:
from datetime import datetime, timezone

EXPORT = (CACHE / "diagnostics" /
          (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
           + f"-route-probe-forget{FORGET_TASK}"))
EXPORT.mkdir(parents=True, exist_ok=True)

summary = {
    "study_id": STUDY_ID, "code_version": CODE_VERSION, "order": ORDER, "seed": SEED,
    "forget_task": FORGET_TASK, "retain_task": RETAIN_TASK,
    "unlearn_budgets": UNLEARN_BUDGETS, "probe_codes": PROBE_CODES,
    "classes_per_task": CLASSES, "margin": MARGIN,
    "results": {str(s): results[s] for s in UNLEARN_BUDGETS},
    "reading": ("Nothing is trained here. Each reading scores the weights that an "
                "unused random task code produces, on one task's test images, using "
                "that task's stored BatchNorm statistics. This repeats the paper's "
                "appendix baseline, which it runs before unlearning, and adds the same "
                "measurement after unlearning. A reading well above chance for the "
                "forgotten task after unlearning means the skill is still reachable "
                "through a different code, so the task was blocked rather than removed. "
                "The kept task is the control: if random codes stop reading it too, the "
                "generator was damaged in general and the probe decides nothing."),
}
(EXPORT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Written to", EXPORT)

if SAVE_TO_R2:
    key = PREFIX + "/diagnostics/" + EXPORT.name + "/summary.json"
    store.upload(EXPORT / "summary.json", key)
    print("Uploaded to R2:", key)